In [2]:
# ============== KAGGLE NOTEBOOK UCHUN TO'LIQ KOD ==============

# Avval barcha kerakli kutubxonalarni import qilish
import os
import shutil
import random
import numpy as np
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
from torch.cuda.amp import autocast, GradScaler

import matplotlib.pyplot as plt
import seaborn as sns

# GPU tekshirish
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ============== 1. DATASET TAYYORLASH ==============
def prepare_dataset():
    """Kaggle datasetini train/val ga bo'lish"""
    base_dir = "flowers"
    output_dir = "/kaggle/working/flowers_split"
    
    print("Dataset tayyorlanmoqda...")
    print(f"Dataset joylashuvi: {base_dir}")
    
    # Avval mavjud papkalarni tekshirish
    if os.path.exists(base_dir):
        print(f"Topilgan sinflar: {os.listdir(base_dir)}")
    else:
        print("❌ Dataset topilmadi! Dataset qo'shilganini tekshiring.")
        return None
    
    # Train va val papkalarini yaratish
    for split in ['train', 'val']:
        os.makedirs(f"{output_dir}/{split}", exist_ok=True)
    
    # Har bir sinf uchun
    total_images = 0
    for class_name in os.listdir(base_dir):
        class_path = os.path.join(base_dir, class_name)
        if not os.path.isdir(class_path):
            continue
            
        # Train va val papkalari yaratish
        os.makedirs(f"{output_dir}/train/{class_name}", exist_ok=True)
        os.makedirs(f"{output_dir}/val/{class_name}", exist_ok=True)
        
        # Rasmlarni olish
        images = [img for img in os.listdir(class_path) if img.endswith(('.jpg', '.png', '.jpeg'))]
        train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)
        
        print(f"📁 {class_name}: train={len(train_imgs)}, val={len(val_imgs)}")
        total_images += len(images)
        
        # Train rasmlarini ko'chirish
        for img in train_imgs:
            src = f"{class_path}/{img}"
            dst = f"{output_dir}/train/{class_name}/{img}"
            shutil.copy(src, dst)
            
        # Val rasmlarini ko'chirish
        for img in val_imgs:
            src = f"{class_path}/{img}"
            dst = f"{output_dir}/val/{class_name}/{img}"
            shutil.copy(src, dst)
    
    print(f"\n✅ Dataset tayyor!")
    print(f"📊 Jami rasmlar: {total_images}")
    print(f"📂 Saqlangan joy: {output_dir}")
    return output_dir

# Dataset tayyorlash
DATA_DIR = prepare_dataset()

if DATA_DIR is None:
    print("Dataset topilmadi! Dataset qo'shilganligini tekshiring.")
else:
    # ============== 2. HYPERPARAMETERS ==============
    NUM_CLASSES = 5
    BATCH = 32
    EPOCHS_HEAD = 3   # Faqat head o'qitish
    EPOCHS_FULL = 7   # Butun model o'qitish
    EPOCHS_RES = 10   # ResNet uchun
    LR_HEAD = 3e-4
    LR_FULL = 1e-4
    LR_RES = 3e-4
    DEVICE = device
    SEED = 42
    NUM_WORKERS = 2
    USE_AMP = True

    print(f"\n🔧 Hyperparameters:")
    print(f"  Batch size: {BATCH}")
    print(f"  ViT epochs: {EPOCHS_HEAD + EPOCHS_FULL}")
    print(f"  ResNet epochs: {EPOCHS_RES}")

    # ============== 3. SEED SOZLASH ==============
    def set_seed(sd=SEED):
        random.seed(sd)
        np.random.seed(sd)
        torch.manual_seed(sd)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(sd)

    set_seed()

    # ============== 4. DATA TRANSFORMS ==============
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)

    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    val_tfms = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    # ============== 5. DATALOADERS ==============
    train_ds = datasets.ImageFolder(f"{DATA_DIR}/train", transform=train_tfms)
    val_ds = datasets.ImageFolder(f"{DATA_DIR}/val", transform=val_tfms)

    train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True, 
                          num_workers=NUM_WORKERS, pin_memory=True)
    val_ld = DataLoader(val_ds, batch_size=BATCH, shuffle=False, 
                        num_workers=NUM_WORKERS, pin_memory=True)

    idx_to_class = {v: k for k, v in train_ds.class_to_idx.items()}
    print(f"\n📋 Dataset info:")
    print(f"  Sinflar: {list(idx_to_class.values())}")
    print(f"  Train: {len(train_ds)} ta rasm")
    print(f"  Val: {len(val_ds)} ta rasm")

    # ============== 6. LOSS FUNCTION ==============
    criterion = nn.CrossEntropyLoss()

    # ============== 7. TRAINING FUNCTIONS ==============
    def train_one_epoch(model, loader, optimizer, use_amp=USE_AMP):
        model.train()
        scaler = GradScaler(enabled=use_amp)
        tot_loss, correct, total = 0.0, 0, 0
        
        for batch_idx, (x, y) in enumerate(loader):
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            
            with autocast(enabled=use_amp):
                out = model(x)
                loss = criterion(out, y)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            tot_loss += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
        
        return tot_loss/total, correct/total

    @torch.no_grad()
    def evaluate(model, loader):
        model.eval()
        tot_loss, correct, total = 0.0, 0, 0
        all_preds, all_trues = [], []
        
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out, y)
            
            tot_loss += loss.item() * y.size(0)
            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
            
            all_preds.append(pred.cpu().numpy())
            all_trues.append(y.cpu().numpy())
        
        return tot_loss/total, correct/total, np.concatenate(all_preds), np.concatenate(all_trues)

    def log_epoch(tag, ep, tr, va):
        print(f"{tag} Epoch {ep:02d} | Train Loss: {tr[0]:.4f}, Acc: {tr[1]:.3f} | Val Loss: {va[0]:.4f}, Acc: {va[1]:.3f}")

    # ============== 8. VISION TRANSFORMER (ViT) ==============
    print("\n" + "="*60)
    print("🤖 VISION TRANSFORMER (ViT) TRAINING")
    print("="*60)

    # ViT modelini yuklash
    print("ViT model yuklanmoqda...")
    vit = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    vit.heads.head = nn.Linear(vit.heads.head.in_features, NUM_CLASSES)
    vit = vit.to(DEVICE)
    print(f"✓ ViT-B/16 model yuklandi")

    # STEP 1: Faqat head'ni o'qitish
    print("\n>>> STEP 1: Head-only training (freezing backbone)")
    for p in vit.parameters():
        p.requires_grad = False
    for p in vit.heads.parameters():
        p.requires_grad = True

    opt_vit = optim.AdamW(filter(lambda p: p.requires_grad, vit.parameters()), 
                          lr=LR_HEAD, weight_decay=0.05)

    vit_train_losses, vit_train_accs = [], []
    vit_val_losses, vit_val_accs = [], []

    for e in range(1, EPOCHS_HEAD + 1):
        tr = train_one_epoch(vit, train_ld, opt_vit)
        va = evaluate(vit, val_ld)
        log_epoch("[ViT-HEAD]", e, tr, va)
        vit_train_losses.append(tr[0])
        vit_train_accs.append(tr[1])
        vit_val_losses.append(va[0])
        vit_val_accs.append(va[1])

    # STEP 2: Butun modelni o'qitish
    print("\n>>> STEP 2: Full model fine-tuning")
    for p in vit.parameters():
        p.requires_grad = True

    opt_vit = optim.AdamW(vit.parameters(), lr=LR_FULL, weight_decay=0.05)

    best_vit_acc = 0.0
    best_vit_path = "/kaggle/working/vit_b16_best.pth"

    for e in range(1, EPOCHS_FULL + 1):
        tr = train_one_epoch(vit, train_ld, opt_vit)
        va = evaluate(vit, val_ld)
        log_epoch("[ViT-FULL]", e, tr, va)
        vit_train_losses.append(tr[0])
        vit_train_accs.append(tr[1])
        vit_val_losses.append(va[0])
        vit_val_accs.append(va[1])
        
        if va[1] > best_vit_acc:
            best_vit_acc = va[1]
            torch.save(vit.state_dict(), best_vit_path)
            print(f"  ✓ Best ViT model saved! Accuracy: {best_vit_acc:.3f}")

    # ============== 9. RESNET18 (CNN) ==============
    print("\n" + "="*60)
    print("🏛️ RESNET18 (CNN) TRAINING")
    print("="*60)

    print("ResNet18 model yuklanmoqda...")
    res = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    res.fc = nn.Linear(res.fc.in_features, NUM_CLASSES)
    res = res.to(DEVICE)
    print(f"✓ ResNet18 model yuklandi")

    opt_res = optim.Adam(res.parameters(), lr=LR_RES, weight_decay=1e-4)

    best_res_acc = 0.0
    best_res_path = "/kaggle/working/resnet18_best.pth"

    res_train_losses, res_train_accs = [], []
    res_val_losses, res_val_accs = [], []

    for e in range(1, EPOCHS_RES + 1):
        tr = train_one_epoch(res, train_ld, opt_res)
        va = evaluate(res, val_ld)
        log_epoch("[ResNet18]", e, tr, va)
        res_train_losses.append(tr[0])
        res_train_accs.append(tr[1])
        res_val_losses.append(va[0])
        res_val_accs.append(va[1])
        
        if va[1] > best_res_acc:
            best_res_acc = va[1]
            torch.save(res.state_dict(), best_res_path)
            print(f"  ✓ Best ResNet model saved! Accuracy: {best_res_acc:.3f}")

    # ============== 10. FINAL EVALUATION ==============
    print("\n" + "="*60)
    print("📊 FINAL EVALUATION")
    print("="*60)

    # Best modellarni yuklash
    vit.load_state_dict(torch.load(best_vit_path, map_location=DEVICE))
    res.load_state_dict(torch.load(best_res_path, map_location=DEVICE))

    # Evaluation
    _, vit_acc, vit_pred, vit_true = evaluate(vit, val_ld)
    _, res_acc, res_pred, res_true = evaluate(res, val_ld)

    print(f"\n>>> VALIDATION RESULTS:")
    print(f"ViT-B/16  Accuracy: {vit_acc:.3f} ({vit_acc*100:.1f}%)")
    print(f"ResNet-18 Accuracy: {res_acc:.3f} ({res_acc*100:.1f}%)")
    print(f"🏆 Winner: {'ViT' if vit_acc > res_acc else 'ResNet18'} (+{abs(vit_acc - res_acc)*100:.1f}%)")

    # Classification reports
    names = [idx_to_class[i] for i in range(NUM_CLASSES)]

    print("\n" + "-"*50)
    print("ViT CLASSIFICATION REPORT:")
    print("-"*50)
    print(classification_report(vit_true, vit_pred, target_names=names, digits=3))

    print("\n" + "-"*50)
    print("RESNET18 CLASSIFICATION REPORT:")
    print("-"*50)
    print(classification_report(res_true, res_pred, target_names=names, digits=3))

    # ============== 11. VISUALIZATION ==============
    print("\n" + "="*60)
    print("📈 GENERATING VISUALIZATIONS")
    print("="*60)

    # Training curves
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # ViT Loss
    axes[0, 0].plot(vit_train_losses, label='Train Loss', color='blue', alpha=0.7)
    axes[0, 0].plot(vit_val_losses, label='Val Loss', color='red', alpha=0.7)
    axes[0, 0].set_title('ViT Training/Validation Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # ViT Accuracy
    axes[0, 1].plot(vit_train_accs, label='Train Acc', color='green', alpha=0.7)
    axes[0, 1].plot(vit_val_accs, label='Val Acc', color='orange', alpha=0.7)
    axes[0, 1].set_title('ViT Training/Validation Accuracy')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # ResNet Loss
    axes[1, 0].plot(res_train_losses, label='Train Loss', color='blue', alpha=0.7)
    axes[1, 0].plot(res_val_losses, label='Val Loss', color='red', alpha=0.7)
    axes[1, 0].set_title('ResNet18 Training/Validation Loss')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # ResNet Accuracy
    axes[1, 1].plot(res_train_accs, label='Train Acc', color='green', alpha=0.7)
    axes[1, 1].plot(res_val_accs, label='Val Acc', color='orange', alpha=0.7)
    axes[1, 1].set_title('ResNet18 Training/Validation Accuracy')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Accuracy')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Confusion Matrices
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # ViT Confusion Matrix
    cm_vit = confusion_matrix(vit_true, vit_pred)
    sns.heatmap(cm_vit, annot=True, fmt='d', cmap='Blues', 
                xticklabels=names, yticklabels=names, ax=axes[0])
    axes[0].set_title(f'ViT Confusion Matrix (Acc: {vit_acc:.3f})')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')

    # ResNet Confusion Matrix
    cm_res = confusion_matrix(res_true, res_pred)
    sns.heatmap(cm_res, annot=True, fmt='d', cmap='Greens',
                xticklabels=names, yticklabels=names, ax=axes[1])
    axes[1].set_title(f'ResNet18 Confusion Matrix (Acc: {res_acc:.3f})')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')

    plt.tight_layout()
    plt.show()

    print("\n✅ Training completed successfully!")
    print("\n📊 Final Results:")
    print(f"• ViT model accuracy: {vit_acc:.1%}")
    print(f"• ResNet model accuracy: {res_acc:.1%}")
    print(f"• Best model: {'ViT' if vit_acc > res_acc else 'ResNet18'}")


Device: cpu
Dataset tayyorlanmoqda...
Dataset joylashuvi: flowers
Topilgan sinflar: ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']
📁 daisy: train=611, val=153
📁 dandelion: train=841, val=211
📁 rose: train=627, val=157
📁 sunflower: train=586, val=147
📁 tulip: train=787, val=197

✅ Dataset tayyor!
📊 Jami rasmlar: 4317
📂 Saqlangan joy: /kaggle/working/flowers_split

🔧 Hyperparameters:
  Batch size: 32
  ViT epochs: 10
  ResNet epochs: 10

📋 Dataset info:
  Sinflar: ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']
  Train: 3452 ta rasm
  Val: 865 ta rasm

🤖 VISION TRANSFORMER (ViT) TRAINING
ViT model yuklanmoqda...
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to C:\Users\behru/.cache\torch\hub\checkpoints\vit_b_16-c867db91.pth


100.0%


✓ ViT-B/16 model yuklandi

>>> STEP 1: Head-only training (freezing backbone)


C:\Users\behru\AppData\Local\Temp\ipykernel_21516\1850853774.py:158: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
c:\Users\behru\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\amp\grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
c:\Users\behru\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\behru\AppData\Local\Temp\ipykernel_21516\1850853774.py:165: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
c:\Users\behru\AppData\Local\Programs\Python\Python312\Lib\site-packages\t

KeyboardInterrupt: 